In [1]:
from pyspark.sql.session import SparkSession


spark = (

    SparkSession.builder.getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/13 10:53:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Splittable File Types and Compression


Certain file formats are fundamentally “splittable.” This can improve speed because it makes it
possible for Spark to avoid reading an entire file, and access only the parts of the file necessary

to satisfy your query. Additionally if you’re using something like Hadoop Distributed File
System (HDFS), 

splitting a file can provide further optimization if that file spans multiple
blocks. In conjunction with this is a need to manage compression. Not all compression schemes
are splittable. How you store your data is of immense consequence when it comes to making
your Spark jobs run smoothly. We recommend Parquet with gzip compression.

## Writing Data in Parallel

In [22]:
spark.conf.set("spark.sql.adaptive.enabled","false")

In [23]:
csv_df  = (
    spark
    .read
    .option("header","true")
    .csv("../data/flight-data/csv")
)

print("distinct dest_country_name", csv_df.select("DEST_COUNTRY_NAME").distinct().count())

csv_df.rdd.getNumPartitions() , spark.sparkContext.defaultParallelism

distinct dest_country_name 167


(6, 8)

In [24]:

csv_df.write.format("parquet").mode("overwrite").partitionBy("DEST_COUNTRY_NAME").save("../data/tmp/flight-data")

#### you get a list of files inside partions DEST_COUNTRY_NAME="XXXX"

In [ ]:
! ls -lrt ../data/tmp/flight-data | head

total 668
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Brunei
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Cameroon
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Burundi
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Chad
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Cote d%27Ivoire
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Djibouti
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Gibraltar
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Guinea
drwxr-xr-x 2 suanki suanki 4096 Nov 13 11:02 DEST_COUNTRY_NAME=Iraq
ls: write error: Broken pipe


## Reading Data in Parallel

In [28]:
path = "../data/tmp/flight-data"

df = spark.read.parquet(path)


df.rdd.getNumPartitions()

25

In [27]:
df.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [ORIGIN_COUNTRY_NAME#373,count#374,DEST_COUNTRY_NAME#375] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/suanki/Desktop/projects/suanki-pyspark-and-optimization/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<ORIGIN_COUNTRY_NAME:string,count:string>




In [30]:
print( spark.conf.get("spark.sql.files.maxPartitionBytes") )
print( spark.conf.get("spark.sql.files.openCostInBytes") )

134217728b
4194304b


## Reading Partitioned Data and Partition Counts

When you read data back from a folder containing multiple files (like a partitioned Parquet directory), the partition count of the new DataFrame is dictated by the physical layout of the files on disk, not the original DataFrame's parallelism. Here is exactly how Spark calculates this new partition count:

* **Files Created During Write:** The number of files written to disk depends on the number of partitions your DataFrame had at the time of writing. Because your previous DataFrame had 6 partitions and you partitioned the output by `DEST_COUNTRY_NAME` (which has 167 distinct values), each of those 6 tasks wrote its own independent Parquet files into the respective 167 country folders. This means Spark generated anywhere from 167 files (best case) up to 1,002 files (6 tasks × 167 countries).
* **Partitions Created During Read:** In general, when you read from a folder with multiple files, each individual file becomes one partition in your new DataFrame. If Spark generated 1,002 Parquet files, `getNumPartitions()` would output 1,002.
* **Small File Packing Optimization:** If your dataset is small and those 1,000+ Parquet files are tiny, Spark may not actually create 1,000+ partitions. Spark SQL uses a configuration called `spark.sql.files.maxPartitionBytes` (which defaults to 128 MB) combined with `spark.sql.files.openCostInBytes` to estimate the cost of opening files. Spark uses these settings to automatically pack multiple small files into a single partition to improve performance. 

*Note: If you want to control the exact number of output files (and thus the future read partitions) when writing partitioned data, you should repartition the data based on your partition column before saving it (e.g., `df.repartition(col("DEST_COUNTRY_NAME")).write...`).*

In [31]:
spark.stop()